In [ ]:
import numpy as np
from scipy import optimize
import matplotlib.pyplot as plt
import pandas as pd
from scipy.interpolate import interp1d

# Molar masses (g/mol) of relevant gas species
mu_dict = {'SiO2': 60.08,
            'TiO2': 79.87,
            'Al2O3': 101.96,
            'FeO_star': 71.84,
            'FeO': 71.84,
            'Fe2O3': 159.69,
            'MnO': 70.94,
            'MgO': 40.30,
            'CaO': 56.08,
            'Na2O': 61.98,
            'K2O': 94.20,
            'P2O5': 141.94,
            'S': 32.06,
            'C': 12.01,
            'H': 1.008,
            'O': 16.00,
            'H2O': 18.01528,
            'H2': 2.01588,
            'H2S': 34.08088,
            'CO2': 44.0095,
            'CO': 28.0101,
            'CH4': 16.04246,
            'SO2': 64.066,
            'S2': 64.142
           }

In [ ]:
def calc_FeOratio(T, P, fO2, x_dict, T0 = 1673.0):
    # INPUT P IN BARS
    # Convert it to Pa now
    P = P * 1e5 # Pa

    # Table 7 parameters
    a  = 0.196
    b  = 1.1492e4   # K
    c  = -6.675
    d_Al2O3 = -2.243
    d_FeO_star   =  -1.828
    d_CaO   =  3.201
    d_Na2O  =  5.854
    d_K2O   =  6.215
    e  = -3.36
    f  = -7.01e-7      # K Pa^-1
    g  = -1.54e-10     # Pa^-1
    h  =  3.85e-17     # K Pa^-2

    # Composition term without FeO: Σ d_i X_i  (i ≠ FeO)
    comp = (
        d_Al2O3 * x_dict.get("Al2O3", 0.0) +
        d_FeO_star * x_dict.get("FeO_star", 0.0) +
        d_CaO   * x_dict.get("CaO",   0.0) +
        d_Na2O  * x_dict.get("Na2O",  0.0) +
        d_K2O   * x_dict.get("K2O",   0.0)
    )

    # P–T terms
    bracket = 1.0 - (T0 / T) - np.log(T / T0)
    PT_term = (f * (P / T)) + (g * ((T - T0) / T) * P) + (h * (P**2 / T))

    # Full equation
    ratio_Fe2O3_to_FeO = np.exp((a*np.log(fO2)) + (b/T) + c + comp + (e*bracket) + PT_term)

    return ratio_Fe2O3_to_FeO

def calc_CO2_solubility_constant(P, T, x_FeO, x_dict):

    mu_magma = 64.52 # Molar mass of magma in g/mol, based on Mt. Etna basalt
    mu_co3 = 60.01 # Molar mass of CO3 in g/mol

    C_CO2 = 0.14
    B_CO2 = -5.3
    b_CO2 = 15.8
    d_Al2O3 = 3.8
    d_FeO = -16.3
    d_Na2O = 20.1

    NBO_O = (2*(x_dict["K2O"]+x_dict["Na2O"]+x_dict["CaO"]+x_dict["MgO"]+x_FeO-x_dict["Al2O3"]))/ \
        (2*x_dict["SiO2"]+2*x_dict["TiO2"]+3*x_dict["Al2O3"]+x_dict["MgO"]+x_FeO+x_dict["CaO"]+x_dict["Na2O"]+x_dict["K2O"])
    
    extra_term = (d_Al2O3 * x_dict.get("Al2O3", 0.0)/(x_dict["CaO"] + x_dict["K2O"] + x_dict["Na2O"])) + \
                 (d_FeO * (x_FeO + x_dict["MgO"])) +  (d_Na2O * (x_dict["Na2O"] + x_dict["K2O"]))

    S1 = np.log(mu_magma/(mu_co3*1e6)) + (C_CO2*P/T) + B_CO2 + (b_CO2*NBO_O) + extra_term # CO2

    return S1


def calc_H2O_solubility_constant(P, T, x_FeO, x_dict):

    mu_magma = 64.52 # Molar mass of magma in g/mol, based on Mt. Etna basalt
    mu_h2o = 18.01528 # Molar mass of H2O in g

    a_H2O = 0.54
    b_H2O = 1.24
    B_H2O = -2.95
    C_H2O = 0.02

    NBO_O = (2*(x_dict["K2O"]+x_dict["Na2O"]+x_dict["CaO"]+x_dict["MgO"]+x_FeO-x_dict["Al2O3"]))/ \
        (2*x_dict["SiO2"]+2*x_dict["TiO2"]+3*x_dict["Al2O3"]+x_dict["MgO"]+x_FeO+x_dict["CaO"]+x_dict["Na2O"]+x_dict["K2O"])

    S2 = np.log(mu_magma/(mu_h2o*100)) + (C_H2O*P/T) + B_H2O + (b_H2O*NBO_O) # H2O

    return S2

def calc_sulfate_ratio(Fe2O3_FeO_ratio, T):
    # SULFATE CHANGE: estimate melt sulfate/sulfide using a Nash-2019-style redox relation.
    fe3_fe2 = max(2.0 * Fe2O3_FeO_ratio, np.finfo(float).tiny)
    return 10 ** (8*np.log10(fe3_fe2) + (8.7436e6 / T**2) - (27703 / T) + 20.273)

In [ ]:
def calc_outgassing(T, mH2O_sat, mCO2_sat, mS_sat, FMQ_sat, alphaG_sat, FeO_star_wt, verbose=False):

    ### Initialize the magma composition

    mass_frac_dict = {
           'SiO2': 47.95 * 1e-2, # wt% * 1e-2
           'TiO2': 1.67 * 1e-2, # wt% * 1e-2
           'Al2O3': 17.32 * 1e-2, # wt% * 1e-2
           'MnO': 0 * 1e-2, # wt% * 1e-2
           'MgO': 5.76 * 1e-2, # wt% * 1e-2
           'CaO': 10.93 * 1e-2, # wt% * 1e-2
           'Na2O': 3.45 * 1e-2, # wt% * 1e-2
           'K2O': 1.99 * 1e-2, # wt% * 1e-2
           'P2O5': 0.51 * 1e-2, # wt% * 1e-2
           'H2O': mH2O_sat, # ppm * 1e-6
            'CO2': mCO2_sat, # ppm * 1e-6
            'S': mS_sat, # ppm * 1e-6
            'FeO_star': FeO_star_wt, # wt% * 1e-2
            'FeO': 0.0, # Placeholder, will be calculated later
            'Fe2O3': 0.0 # Placeholder, will be calculated later
           }

    mu_magma = 1/sum([mass_frac_dict[oxide]/mu_dict[oxide] for oxide in mass_frac_dict.keys()])
    if verbose:
        print("Magma molar mass: ", mu_magma)

    mole_frac_dict = {
        'SiO2':     mass_frac_dict['SiO2']     * mu_magma / mu_dict['SiO2'],
        'TiO2':     mass_frac_dict['TiO2']     * mu_magma / mu_dict['TiO2'],
        'Al2O3':    mass_frac_dict['Al2O3']    * mu_magma / mu_dict['Al2O3'],
        'FeO_star': mass_frac_dict['FeO_star'] * mu_magma / mu_dict['FeO'],
        'FeO':      mass_frac_dict['FeO']      * mu_magma / mu_dict['FeO'],
        'Fe2O3':    mass_frac_dict['Fe2O3']    * mu_magma / mu_dict['Fe2O3'],
        'MnO':      mass_frac_dict['MnO']      * mu_magma / mu_dict['MnO'],
        'MgO':      mass_frac_dict['MgO']      * mu_magma / mu_dict['MgO'],
        'CaO':      mass_frac_dict['CaO']      * mu_magma / mu_dict['CaO'],
        'Na2O':     mass_frac_dict['Na2O']     * mu_magma / mu_dict['Na2O'],
        'K2O':      mass_frac_dict['K2O']      * mu_magma / mu_dict['K2O'],
        'P2O5':     mass_frac_dict['P2O5']     * mu_magma / mu_dict['P2O5'],
        'H2O':      mass_frac_dict['H2O']      * mu_magma / mu_dict['H2O'],
        'CO2':      mass_frac_dict['CO2']      * mu_magma / mu_dict['CO2'],
        'S':        mass_frac_dict['S']        * mu_magma / mu_dict['S'],
    }

    xO_oxides = mole_frac_dict['SiO2']*2 + mole_frac_dict['TiO2']*2 + mole_frac_dict['Al2O3']*3 + \
                mole_frac_dict['MnO'] + mole_frac_dict['MgO'] + mole_frac_dict['CaO'] + \
                mole_frac_dict['Na2O'] + mole_frac_dict['K2O'] + mole_frac_dict['P2O5']*5

    ### Get mole fractions
    xH2O_sat = mole_frac_dict['H2O']
    xCO2_sat = mole_frac_dict['CO2']
    xS_sat   = mole_frac_dict['S']

    ### Equilibrium constants for speciation reactions
    K1 = np.exp(-29755.11319228574/T+6.652127716162998) # H2 + 0.5O2 = H2O
    K2 = np.exp(-33979.12369002451/T+10.418882755464773) # CO + 0.5O2 = CO2
    K3 = np.exp(-96444.47151911151/T+0.22260815074146403) # CH4 + 2O2 = CO2 + 2H2O
    K4 = np.exp(4.35250424e+04/T-8.80403494e+00) # 0.5S2 + O2 = SO2
    K5 = np.exp(1.90560415e+04/T-8.60366131e-01) # H2S + 0.5O2 = 0.5S2 + H2O

    def nan_results():
        nan_arr = np.full_like(np.logspace(3, -5, num=9), np.nan, dtype=float)
        return {
            'P': nan_arr.copy(),
            'H2O': nan_arr.copy(),
            'H2': nan_arr.copy(),
            'CO2': nan_arr.copy(),
            'CO': nan_arr.copy(),
            'CH4': nan_arr.copy(),
            'S2': nan_arr.copy(),
            'SO2': nan_arr.copy(),
            'H2S': nan_arr.copy(),
            'O2': nan_arr.copy(),
            'xCO2': nan_arr.copy(),
            'xH2O': nan_arr.copy(),
            'xH2': nan_arr.copy(),
            'xS2': nan_arr.copy(),
            'xS6': nan_arr.copy(),
            'xS': nan_arr.copy(),
            'alphaG': nan_arr.copy(),
            'FMQ': nan_arr.copy(),
            'xCO2_sat': np.nan,
            'xH2O_sat': np.nan,
            'xH2_sat': np.nan,
            'xS2_sat': np.nan,
            'xS6_sat': np.nan,
            'xS_sat': np.nan,
            'mu_magma': np.nan
        }

    ### Calculate gas-melt composition at saturation

    P_sat = 1000 # bar, The while loop is to converge on P_sat
    max_sat_iter = 200
    for sat_iter in range(max_sat_iter):
        ### Calculate actual fO2 at saturation
        A_fug = 25096.3; B_fug = 8.735; C_fug = 0.11 # FMQ fugacity from VolFe
        log_FMQ_sat = (-A_fug/T+B_fug+C_fug*(P_sat-1)/T)
        fO2_sat = 10**(log_FMQ_sat+FMQ_sat)

        ### Calculate FeO and Fe2O3 mole fractions at saturation
        Fe2O3_FeO_ratio = calc_FeOratio(T, P_sat, fO2_sat, mole_frac_dict)
        x_FeO_sat = mole_frac_dict["FeO_star"] / (1 + 2*Fe2O3_FeO_ratio)
        x_Fe2O3_sat = mole_frac_dict["FeO_star"] /((1/Fe2O3_FeO_ratio) + 2)

        ### Calculate sulfur solubility at saturation
        m_FeO_pct = 100 * x_FeO_sat * mu_dict['FeO'] / mu_magma # Convert mole fraction to wt%
        Cs = (mu_magma/mu_dict['S'])*1e-6*0.0003*(100 - m_FeO_pct)*np.exp(0.21*m_FeO_pct)
        sulfate_ratio_sat = calc_sulfate_ratio(Fe2O3_FeO_ratio, T)
        xS2_sat = xS_sat / (1 + sulfate_ratio_sat)
        xS6_sat = xS_sat - xS2_sat

        d_H2O = 2.3; a_CO2 = 1; a_H2O = 0.54 # Constants
        P_CO2_sat = np.exp((np.log(xCO2_sat) - (xH2O_sat * d_H2O) - calc_CO2_solubility_constant(P_sat, T, x_FeO_sat, mole_frac_dict))/a_CO2)
        P_H2O_sat = np.exp((np.log(xH2O_sat) - calc_H2O_solubility_constant(P_sat, T, x_FeO_sat, mole_frac_dict))/a_H2O) ## From Wogan 2020
        P_S2_sat  = (fO2_sat*xS2_sat**2)/(Cs**2)

        ### Gas composition at saturation
        P_H2_sat = K1*P_H2O_sat/((fO2_sat)**0.5)
        P_CO_sat = K2*P_CO2_sat/((fO2_sat)**0.5)
        P_CH4_sat = K3*P_CO2_sat*(P_H2O_sat**2)/(fO2_sat**2)
        P_SO2_sat = K4*fO2_sat*(P_S2_sat**0.5)
        P_H2S_sat = (P_S2_sat**0.5)*P_H2O_sat/(K5*(fO2_sat**0.5))
        # xH2_sat = np.exp(1.28*np.log(P_H2_sat) + np.log(mu_magma*3.4e-7/(2.016*2.7)))


        P_calc = P_CO2_sat + P_H2O_sat + P_S2_sat + P_H2_sat + P_H2S_sat + P_SO2_sat + P_CO_sat + P_CH4_sat
        if not np.isfinite(P_calc) or P_calc <= 0:
            print(f"Warning: Encountered non-physical saturation pressure ({P_calc}). Returning NaNs.")
            return nan_results()
        if abs(P_calc - P_sat) < 1:
            P_sat = P_calc
            break
        P_sat = P_calc
    else:
        if verbose:
            print(f"Warning: Saturation pressure did not converge after {max_sat_iter} iterations. Returning NaNs.")
        return nan_results()

    ### Calculate total elemental abundances at saturation    
    xC_tot = (((P_CO2_sat + P_CO_sat + P_CH4_sat)/P_sat)*alphaG_sat) + ((1 - alphaG_sat)*(xCO2_sat)) # total C
    xH_tot = (((2*P_H2O_sat + 4*P_CH4_sat + 2*P_H2S_sat + 2*P_H2_sat)/P_sat)*alphaG_sat) + ((1 - alphaG_sat)*(2*xH2O_sat)) # total H
    xS_tot = ((2*P_S2_sat + P_SO2_sat + P_H2S_sat)/P_sat)*alphaG_sat + ((1 - alphaG_sat)*(xS2_sat + xS6_sat)) # total S
    xO_tot = (((P_H2O_sat + 2*P_SO2_sat + 2*P_CO2_sat + P_CO_sat)/P_sat)*alphaG_sat) + ((1 - alphaG_sat)*(1*xH2O_sat + 2*xCO2_sat + 1*x_FeO_sat + 3*x_Fe2O3_sat + 0*xS6_sat + 0*mole_frac_dict["FeO_star"])) # total O

    P_grid = np.logspace(3, -6, num=30) # bar

    P_H2O_arr = np.full_like(P_grid, np.nan, dtype=float); P_H2_arr = np.full_like(P_grid, np.nan, dtype=float); P_CO2_arr = np.full_like(P_grid, np.nan, dtype=float); P_CO_arr = np.full_like(P_grid, np.nan, dtype=float)
    P_CH4_arr = np.full_like(P_grid, np.nan, dtype=float); P_SO2_arr = np.full_like(P_grid, np.nan, dtype=float); P_H2S_arr = np.full_like(P_grid, np.nan, dtype=float); P_S2_arr = np.full_like(P_grid, np.nan, dtype=float)
    xCO2_arr = np.full_like(P_grid, np.nan, dtype=float); xH2O_arr = np.full_like(P_grid, np.nan, dtype=float); xH2_arr = np.full_like(P_grid, np.nan, dtype=float); xS2_arr = np.full_like(P_grid, np.nan, dtype=float); xS6_arr = np.full_like(P_grid, np.nan, dtype=float); xS_arr = np.full_like(P_grid, np.nan, dtype=float)
    fO2_arr = np.full_like(P_grid, np.nan, dtype=float); alphaG_arr = np.full_like(P_grid, np.nan, dtype=float); P_check_arr = np.full_like(P_grid, np.nan, dtype=float); FMQ_final_arr = np.full_like(P_grid, np.nan, dtype=float)

    last_success_idx = None

    for i, P in enumerate(P_grid):

        def system(y):

            ln_P_H2O, ln_P_H2, ln_P_CO2, ln_P_CO, ln_P_CH4, ln_P_S2, ln_P_H2S, ln_P_SO2, ln_fO2, ln_xCO2_melt, ln_xH2O_melt, ln_xS2_melt, alphaG = y

            ### Gas speciation equation
            eq1 = np.log(K1) + ln_P_H2O - 0.5*ln_fO2 - ln_P_H2 # H2 + 0.5O2 = H2O
            eq2 = np.log(K2) + ln_P_CO2 - 0.5*ln_fO2 - ln_P_CO # CO + 0.5O2 = CO2
            eq3 = np.log(K3) + ln_P_CO2 + 2*ln_P_H2O - 2*ln_fO2 - ln_P_CH4 # CH4 + 2O2 = CO2 + 2H2O
            eq4 = np.log(K4) + ln_fO2 + 0.5*ln_P_S2 - ln_P_SO2  # 0.5S2 + O2 = SO2
            eq5 = np.log(K5) + 0.5*ln_fO2 + ln_P_H2S - 0.5*ln_P_S2 - ln_P_H2O # H2S + 0.5O2 = 0.5S2 + H2O

            ### Melt solubility equations
            ### Calculate FeO and Fe2O3 mole fractions
            Fe2O3_FeO_ratio = calc_FeOratio(T, P, np.exp(ln_fO2), mole_frac_dict)
            x_FeO = mole_frac_dict["FeO_star"] / (1 + 2*Fe2O3_FeO_ratio)
            x_Fe2O3 = mole_frac_dict["FeO_star"] /((1/Fe2O3_FeO_ratio) + 2)
            m_FeO_pct = 100*x_FeO * mu_dict['FeO'] / mu_magma  # Convert mole fraction to wt%
            Cs = (mu_magma/mu_dict['S'])*1e-6*0.0003*(100 - m_FeO_pct)*np.exp(0.21*m_FeO_pct)

            # xH2_melt = np.exp(1.28*ln_P_H2 + np.log(mu_magma*3.4e-7/(2.016*2.7)))
            sulfate_ratio = calc_sulfate_ratio(Fe2O3_FeO_ratio, T)
            xS2_melt = np.exp(ln_xS2_melt)
            xS6_melt = sulfate_ratio * xS2_melt
            xS_total_melt = xS2_melt + xS6_melt

            # Actual solubility equations
            d_H2O = 2.3; a_CO2 = 1; a_H2O = 0.54 # Constants
            eq6 = ((ln_xCO2_melt - (np.exp(ln_xH2O_melt) * d_H2O) - calc_CO2_solubility_constant(P, T, x_FeO, mole_frac_dict))/a_CO2) - ln_P_CO2
            eq7 = ((ln_xH2O_melt - calc_H2O_solubility_constant(P, T, x_FeO, mole_frac_dict))/a_H2O) - ln_P_H2O ## From Wogan 2020
            eq8 = ln_xS2_melt + 0.5*ln_fO2 - 0.5*ln_P_S2 - np.log(Cs) # dissolved sulfide solubility equation

            ### Conservation of atoms
            eq9 = ((np.exp(ln_P_CO2) + np.exp(ln_P_CO) + np.exp(ln_P_CH4))*alphaG) + (P*(1 - alphaG)*(np.exp(ln_xCO2_melt))) - (P*xC_tot) # total C
            eq10 = ((2*np.exp(ln_P_H2O) + 4*np.exp(ln_P_CH4) + 2*np.exp(ln_P_H2S) + 2*np.exp(ln_P_H2))*alphaG) + (P*(1 - alphaG)*(2*np.exp(ln_xH2O_melt))) - (P*xH_tot) # total H
            eq11 = ((2*np.exp(ln_P_S2) + np.exp(ln_P_SO2) + np.exp(ln_P_H2S))*alphaG) + (P*(1 - alphaG)*xS_total_melt) - (P*xS_tot) # total S
            eq12 = ((np.exp(ln_P_H2O) + 2*np.exp(ln_P_SO2) + 2*np.exp(ln_P_CO2) + np.exp(ln_P_CO))*alphaG) + (P*(1 - alphaG)*(1*np.exp(ln_xH2O_melt) + 2*np.exp(ln_xCO2_melt) + 1*x_FeO + 3*x_Fe2O3 + 0*xS6_melt + 0*mole_frac_dict["FeO_star"])) - (P*xO_tot) # total O

            ### Conservation of total mass
            eq13 = np.exp(ln_P_H2O) + np.exp(ln_P_H2) + np.exp(ln_P_CO2) + np.exp(ln_P_CO) + np.exp(ln_P_CH4) + np.exp(ln_P_S2) + np.exp(ln_P_H2S) + np.exp(ln_P_SO2) - P
            return [eq1, eq2, eq3, eq4, eq5, eq6, eq7, eq8, eq9, eq10, eq11, eq12, eq13]


        ### Build continuation initial guess from the latest successful pressure level.
        eps = np.finfo(float).tiny
        if last_success_idx is None:
            p_ratio = P / P_sat
            init_cond = [np.log(max(P_H2O_sat * p_ratio, eps)), np.log(max(P_H2_sat * p_ratio, eps)), np.log(max(P_CO2_sat * p_ratio, eps)), np.log(max(P_CO_sat * p_ratio, eps)), np.log(max(P_CH4_sat * p_ratio, eps)), np.log(max(P_S2_sat * p_ratio, eps)), 
                        np.log(max(P_H2S_sat * p_ratio, eps)), np.log(max(P_SO2_sat * p_ratio, eps)), np.log(max(fO2_sat, eps)), np.log(max(xCO2_sat, eps)), np.log(max(xH2O_sat, eps)), 
                        np.log(max(xS2_sat, eps)), alphaG_sat]
        else:
            p_ref = P_grid[last_success_idx]
            p_ratio = P / p_ref
            alpha_init = alphaG_arr[last_success_idx] if np.isfinite(alphaG_arr[last_success_idx]) else alphaG_sat
            init_cond = [np.log(max(P_H2O_arr[last_success_idx] * p_ratio, eps)), np.log(max(P_H2_arr[last_success_idx] * p_ratio, eps)), np.log(max(P_CO2_arr[last_success_idx] * p_ratio, eps)), np.log(max(P_CO_arr[last_success_idx] * p_ratio, eps)), np.log(max(P_CH4_arr[last_success_idx] * p_ratio, eps)), np.log(max(P_S2_arr[last_success_idx] * p_ratio, eps)), 
                        np.log(max(P_H2S_arr[last_success_idx] * p_ratio, eps)), np.log(max(P_SO2_arr[last_success_idx] * p_ratio, eps)), np.log(max(fO2_arr[last_success_idx], eps)), np.log(max(xCO2_arr[last_success_idx], eps)), np.log(max(xH2O_arr[last_success_idx], eps)), 
                        np.log(max(xS2_arr[last_success_idx], eps)), alpha_init]

        try:
            sol = optimize.root(system,init_cond,method='lm',options={'maxiter': 1000000}) # Solve the system with the Levenberg-Marquardt algorithm
            residual = np.asarray(system(sol['x']), dtype=float)
            error = np.linalg.norm(residual) # Calculate the error of the solution
            has_non_finite = (not np.all(np.isfinite(residual))) or (not np.isfinite(error)) or (not np.all(np.isfinite(sol['x'])))
        except Exception as exc:
            print(f"Warning: Solver failed at pressure {P:.2f} ({exc}). Setting this level to NaN and continuing.")
            if verbose:
                print("pressure:", P)
            continue

        ### Check if the solution is valid
        tol = 1e-7
        if has_non_finite or error > tol or sol['success'] == False:
            print(f"Warning: Solver did not converge at pressure {P:.2f}. Setting this level to NaN and continuing.")
            if verbose:
                print("success:", sol.success)
                print("status:", sol.status)
                print("message:", sol.message)
                print("error_norm:", error)
                print("has_non_finite:", has_non_finite)
            continue

        ### Extract final results
        ln_P_H2O, ln_P_H2, ln_P_CO2, ln_P_CO, ln_P_CH4, ln_P_S2, ln_P_H2S, ln_P_SO2, ln_fO2, ln_xCO2_melt, ln_xH2O_melt, ln_xS2_melt, alphaG = sol['x']
        P_H2O = np.exp(ln_P_H2O); P_H2 = np.exp(ln_P_H2); P_CO2 = np.exp(ln_P_CO2); P_CO = np.exp(ln_P_CO)
        P_CH4 = np.exp(ln_P_CH4); P_S2 = np.exp(ln_P_S2); P_H2S = np.exp(ln_P_H2S); P_SO2 = np.exp(ln_P_SO2)
        xCO2_melt = np.exp(ln_xCO2_melt); xH2O_melt = np.exp(ln_xH2O_melt); xS2_melt = np.exp(ln_xS2_melt)
        xH2_melt = np.exp(1.28*ln_P_H2 + np.log(mu_magma*3.4e-7/(2.016*2.7)))
        fO2 = np.exp(ln_fO2)
        Fe2O3_FeO_ratio = calc_FeOratio(T, P, fO2, mole_frac_dict)
        sulfate_ratio = calc_sulfate_ratio(Fe2O3_FeO_ratio, T)
        xS6_melt = sulfate_ratio * xS2_melt
        xS_melt = xS2_melt + xS6_melt

        P_check = P_H2O + P_H2 + P_CO2 + P_CO + P_CH4 + P_S2 + P_H2S + P_SO2

        A_fug = 25096.3; B_fug = 8.735; C_fug = 0.11 # FMQ fugacity from VolFe
        log_FMQ = (-A_fug/T+B_fug+C_fug*(P-1)/T)
        FMQ_final = np.log10(fO2) - log_FMQ

        ### Store results in arrays
        P_H2O_arr[i] = P_H2O
        P_H2_arr[i] = P_H2
        P_CO2_arr[i] = P_CO2
        P_CO_arr[i] = P_CO
        P_CH4_arr[i] = P_CH4
        P_S2_arr[i] = P_S2
        P_H2S_arr[i] = P_H2S
        P_SO2_arr[i] = P_SO2
        xCO2_arr[i] = xCO2_melt
        xH2O_arr[i] = xH2O_melt
        xH2_arr[i] = xH2_melt
        xS2_arr[i] = xS2_melt
        xS6_arr[i] = xS6_melt
        xS_arr[i] = xS_melt
        fO2_arr[i] = fO2
        alphaG_arr[i] = alphaG
        FMQ_final_arr[i] = FMQ_final
        P_check_arr[i] = P_check
        last_success_idx = i

    results = {
        'P': P_grid,
        'H2O': P_H2O_arr,
        'H2': P_H2_arr,
        'CO2': P_CO2_arr,
        'CO': P_CO_arr,
        'CH4': P_CH4_arr,
        'S2': P_S2_arr,
        'SO2': P_SO2_arr,
        'H2S': P_H2S_arr,
        'O2': fO2_arr,
        'xCO2': xCO2_arr,
        'xH2O': xH2O_arr,
        'xH2': xH2_arr,
        'xS2': xS2_arr,
        'xS6': xS6_arr,
        'xS': xS_arr,
        'alphaG': alphaG_arr,
        'FMQ': FMQ_final_arr,
        'xCO2_sat': xCO2_sat,
        'xH2O_sat': xH2O_sat,
        'xS2_sat': xS2_sat,
        'xS6_sat': xS6_sat,
        'xS_sat': xS_sat,
        'mu_magma': mu_magma
    }

    return results


In [ ]:
def plot_speciation(results):
    P_grid = results['P']
    P_H2O_arr = results['H2O']
    P_H2_arr = results['H2']
    P_CO2_arr = results['CO2']
    P_CO_arr = results['CO']
    P_CH4_arr = results['CH4']
    P_S2_arr = results['S2']
    P_SO2_arr = results['SO2']
    P_H2S_arr = results['H2S']

    plt.figure(figsize=(10,6))
    ax = plt.gca()
    ax.loglog(P_grid, np.divide(P_H2O_arr, P_grid), label='H$_2$O', color='blue')
    ax.loglog(P_grid, np.divide(P_H2_arr, P_grid), label='H$_2$', color='lightblue')
    ax.loglog(P_grid, np.divide(P_CO2_arr, P_grid), label='CO$_2$', color='brown')
    ax.loglog(P_grid, np.divide(P_CO_arr, P_grid), label='CO', color='green')
    ax.loglog(P_grid, np.divide(P_SO2_arr, P_grid), label='SO$_2$', color='red')
    ax.loglog(P_grid, np.divide(P_H2S_arr, P_grid), label='H$_2$S', color='pink')
    ax.loglog(P_grid, np.divide(P_S2_arr, P_grid), label='S$_2$', color='orange')
    ax.set_xlim(1e-6, 1e3)
    ax.set_ylim(1e-3, 1)
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.8), borderaxespad=0, ncol=2)
    ax.set_ylabel('Speciation in COHS gas (by moles)')
    ax.set_xlabel('Total Pressure (bar)')
    ax.grid(True)
    return

In [ ]:
### Outgassing calcuation

T = 1300 + 273.15  # C + 273.15 = K
mH2O_sat = 500 * 1e-6 # magma H2O content at saturation in ppm wt -- actually just H content
mCO2_sat = 200 * 1e-6 # magma CO2 content at saturation in ppm wt -- actually just C content
mS_sat = 1000 * 1e-6 # magma S content at saturation in ppm wt -- just S content
FMQ_sat = +1.5 # fO2 at saturation relative to FMQ buffer
alphaG_sat = 1e-8 # Assume amount of gas in melt at saturation by mol (0.3-0.7 wt% from IM 2012; 0.01 wt% for GS 2009)
FeO_star_wt = 14.99 * 1e-2 # Total FeO in melt

outgassing_precalc = calc_outgassing(T, mH2O_sat, mCO2_sat, mS_sat, FMQ_sat, alphaG_sat, FeO_star_wt, verbose=False)
plot_speciation(outgassing_precalc)

<!-- EVo benchmark comparison -->
## EVo benchmark comparison

This is a minimal comparison block: reuse `outgassing_precalc`, run `EVo` with the same bulk inputs, and overlay the `EVo` gas speciation as dashed lines.


In [ ]:
# Minimal EVo comparison: setup, run, and overlay
import importlib
import os
import sys
from pathlib import Path

repo_runtime = Path('external/evo_runtime')
if repo_runtime.exists():
    repo_runtime_str = str(repo_runtime.resolve())
    if repo_runtime_str not in sys.path:
        sys.path.append(repo_runtime_str)

import yaml

evo_src = Path('external/EVo/src')
if not evo_src.exists():
    raise FileNotFoundError('Expected external/EVo/src to exist.')
evo_src_str = str(evo_src.resolve())
if evo_src_str not in sys.path:
    sys.path.insert(0, evo_src_str)

mpl_dir = Path('external/mplconfig')
mpl_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(mpl_dir.resolve()))

# Force a fresh import so notebook reruns pick up local EVo source edits.
for module_name in sorted([name for name in sys.modules if name == 'evo' or name.startswith('evo.')], reverse=True):
    del sys.modules[module_name]
importlib.invalidate_caches()

import evo
import evo.solvgas as evo_solvgas


if not hasattr(evo_solvgas, '_simple_original_find_Y'):
    evo_solvgas._simple_original_find_Y = evo_solvgas.find_Y

def _simple_find_Y(P, T, species_list):
    # EVo's low-pressure real-gas solve fails for this case, so below 1 bar use ideal-gas coefficients.
    if P < 1.0:
        ordered_species = ['H2O', 'O2', 'H2', 'CO', 'CO2', 'CH4', 'S2', 'SO2', 'H2S', 'N2', 'OCS']
        return tuple(1.0 if species in species_list else 0.0 for species in ordered_species)
    return evo_solvgas._simple_original_find_Y(P, T, species_list)

evo_solvgas.find_Y = _simple_find_Y

def run_evo_simple(T, mH2O_sat, mCO2_sat, mS_sat, FMQ_sat, FeO_star_wt):
    feo_star_pct = 100.0 * float(FeO_star_wt)
    workdir_name = (
        f"T{float(T):.2f}_H2O{float(mH2O_sat):.6f}_CO2{float(mCO2_sat):.6f}_"
        f"S{float(mS_sat):.6f}_FMQ{float(FMQ_sat):+.2f}_FeOt{feo_star_pct:.2f}"
    ).replace('+', 'plus').replace('-', 'minus').replace('.', 'p')
    workdir = Path('data/evo_benchmark_grid') / workdir_name
    workdir.mkdir(parents=True, exist_ok=True)

    chem = {
        'SIO2': 47.95,
        'TIO2': 1.67,
        'AL2O3': 17.32,
        'FEO': feo_star_pct,
        'MNO': 0.0,
        'MGO': 5.76,
        'CAO': 10.93,
        'NA2O': 3.45,
        'K2O': 1.99,
        'P2O5': 0.51,
    }
    env = {
        'COMPOSITION': 'basalt',
        'RUN_TYPE': 'closed',
        'SINGLE_STEP': False,
        'FIND_SATURATION': True,
        'ATOMIC_MASS_SET': False,
        'GAS_SYS': 'COHS',
        'FE_SYSTEM': True,
        'OCS': False,
        'S_SAT_WARN': False,
        'T_START': float(T),
        'P_START': 3000.0,
        'P_STOP': 1e-6,
        'DP_MIN': 1e-6,
        'DP_MAX': 100.0,
        'MASS': 100.0,
        'WgT': 1e-8,
        'LOSS_FRAC': 0.9999,
        'DENSITY_MODEL': 'spera2000',
        'FO2_MODEL': 'kc1991',
        'FMQ_MODEL': 'frost1991',
        'H2O_MODEL': 'burguisser2015',
        'H2_MODEL': 'gaillard2003',
        'C_MODEL': 'burguisser2015',
        'CO_MODEL': 'armstrong2015',
        'CH4_MODEL': 'ardia2013',
        'SULFIDE_CAPACITY': 'oneill2020',
        'SULFATE_CAPACITY': 'nash2019',
        'SCSS': 'liu2007',
        'N_MODEL': 'libourel2003',
        'FO2_buffer_SET': True,
        'FO2_buffer': 'FMQ',
        'FO2_buffer_START': float(FMQ_sat),
        'FO2_SET': False,
        'FO2_START': 0.0,
        'FH2_SET': False,
        'FH2_START': 0.24,
        'FH2O_SET': False,
        'FH2O_START': 1000.0,
        'FCO2_SET': False,
        'FCO2_START': 0.01,
        'ATOMIC_H': 550.0,
        'ATOMIC_C': 200.0,
        'ATOMIC_S': 4000.0,
        'ATOMIC_N': 10.0,
        'WTH2O_SET': True,
        'WTH2O_START': float(mH2O_sat),
        'WTCO2_SET': True,
        'WTCO2_START': float(mCO2_sat),
        'SULFUR_SET': True,
        'SULFUR_START': float(mS_sat),
        'NITROGEN_SET': False,
        'NITROGEN_START': 0.0,
        'GRAPHITE_SATURATED': False,
        'GRAPHITE_START': 0.0,
    }
    output = {
        'plot_melt_species': False,
        'plot_gas_species_wt': False,
        'plot_gas_species_mol': False,
        'plot_gas_fraction': False,
        'plot_fo2_dFMQ': False,
    }

    chem_path = workdir / 'chem.yaml'
    env_path = workdir / 'env.yaml'
    output_path = workdir / 'output.yaml'
    chem_path.write_text(yaml.safe_dump(chem, sort_keys=False))
    env_path.write_text(yaml.safe_dump(env, sort_keys=False))
    output_path.write_text(yaml.safe_dump(output, sort_keys=False))

    return evo.run_evo(str(chem_path), str(env_path), str(output_path), folder=str(workdir / 'outputs')).sort_values('P')


def plot_speciation_evo(results, evo_df):
    from matplotlib.lines import Line2D
    import matplotlib.ticker as mticker

    def fmt_value(label, value, fmt):
        return f"{label} = {format(value, fmt)}" if np.isfinite(value) else f"{label} = ?"

    P_grid = results['P']
    species_styles = [
        ('H$_2$O', 'H2O', 'mH2O', 'blue'),
        ('H$_2$', 'H2', 'mH2', 'lightblue'),
        ('CO$_2$', 'CO2', 'mCO2', 'brown'),
        ('CO', 'CO', 'mCO', 'green'),
        ('SO$_2$', 'SO2', 'mSO2', 'red'),
        ('H$_2$S', 'H2S', 'mH2S', 'pink'),
        ('S$_2$', 'S2', 'mS2', 'orange'),
    ]

    fig, ax = plt.subplots(figsize=(10, 6))
    for label, result_key, evo_key, color in species_styles:
        ax.loglog(P_grid, np.divide(results[result_key], P_grid), color=color)
        ax.loglog(evo_df['P'], evo_df[evo_key], color=color, ls='--')

    temperature = globals().get('T', np.nan)
    delta_fmq = globals().get('FMQ_sat', np.nan)
    h2o_ppm = globals().get('mH2O_sat', np.nan)
    co2_ppm = globals().get('mCO2_sat', np.nan)
    s_ppm = globals().get('mS_sat', np.nan)
    feo_star_wt = globals().get('FeO_star_wt', np.nan)

    temperature_c = temperature - 273.15 if np.isfinite(temperature) else np.nan
    h2o_ppm = h2o_ppm * 1e6 if np.isfinite(h2o_ppm) else np.nan
    co2_ppm = co2_ppm * 1e6 if np.isfinite(co2_ppm) else np.nan
    s_ppm = s_ppm * 1e6 if np.isfinite(s_ppm) else np.nan
    feo_star_pct = feo_star_wt * 100 if np.isfinite(feo_star_wt) else np.nan

    input_text = '\n'.join([
        f'T = {temperature_c:.0f} C' if np.isfinite(temperature_c) else 'T = ?',
        rf'$f_{{\mathrm{{O_2}}}}^{{\mathrm{{sat}}}}$ = ' + f'FMQ{delta_fmq:+.2f}' if np.isfinite(delta_fmq) else r'$f_{\mathrm{O_2}}^{\mathrm{sat}}$ = ?',
        rf'$x_{{\mathrm{{H_2O}}}}^{{\mathrm{{sat}}}}$ = ' + f'{h2o_ppm:.0f} ppm wt' if np.isfinite(h2o_ppm) else r'$x_{\mathrm{H_2O}}^{\mathrm{sat}}$ = ?',
        rf'$x_{{\mathrm{{CO_2}}}}^{{\mathrm{{sat}}}}$ = ' + f'{co2_ppm:.0f} ppm wt' if np.isfinite(co2_ppm) else r'$x_{\mathrm{CO_2}}^{\mathrm{sat}}$ = ?',
        rf'$x_{{\mathrm{{S}}}}^{{\mathrm{{sat}}}}$ = ' + f'{s_ppm:.0f} ppm wt' if np.isfinite(s_ppm) else r'$x_{\mathrm{S}}^{\mathrm{sat}}$ = ?',
        rf'$x_{{\mathrm{{FeO*}}}}$ = ' + f'{feo_star_pct:.2f} wt%' if np.isfinite(feo_star_pct) else r'$x_{\mathrm{FeO*}}$ = ?',
    ])
    ax.text(
        0.02,
        0.03,
        input_text,
        transform=ax.transAxes,
        ha='left',
        va='bottom',
        fontsize=9,
        bbox={'facecolor': 'white', 'edgecolor': '0.7', 'alpha': 0.9, 'boxstyle': 'round,pad=0.3'},
    )

    style_handles = [
        Line2D([0], [0], color='black', lw=2, label='This work'),
        Line2D([0], [0], color='black', lw=2, ls='--', label='Liggins et al. 2020'),
    ]
    species_handles = [Line2D([0], [0], color=color, lw=2, label=label) for label, _, _, color in species_styles]

    ax.set_xlim(1e-6, 1e3)
    ax.set_ylim(1e-3, 1)
    ax.yaxis.set_major_locator(mticker.LogLocator(base=10.0))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda value, _: f'{100 * value:g}%'))
    ax.yaxis.set_minor_formatter(mticker.NullFormatter())
    ax.legend(handles=style_handles + species_handles, loc='center left', bbox_to_anchor=(1.02, 0.8), borderaxespad=0)
    ax.set_ylabel('Speciation in COHS gas (by moles)')
    ax.set_xlabel('Total Pressure (bar)')
    ax.grid(True)
    fig.tight_layout(rect=[0, 0, 0.82, 1])
    return evo_df

try:
    EVo_results = run_evo_simple(T, mH2O_sat, mCO2_sat, mS_sat, FMQ_sat, FeO_star_wt)
except Exception as exc:
    EVo_results = None
    print(f'EVo failed for FMQ={FMQ_sat:.1f}: {exc}')
else:
    plot_speciation_evo(outgassing_precalc, EVo_results)
    EVo_results[['P', 'mH2O', 'mCO2', 'mSO2', 'mH2S', 'mS2']].head()


In [ ]:
def plot_speciation_evo_grid(cases):
    from matplotlib.lines import Line2D
    import matplotlib.ticker as mticker

    if len(cases) != 6:
        raise ValueError('Expected exactly 6 cases for a 2x3 grid.')

    inset_fontsize = 12
    tick_fontsize = 15
    axis_label_fontsize = 19
    legend_fontsize = 15

    species_styles = [
        ('H$_2$O', 'H2O', 'mH2O', 'blue'),
        ('H$_2$', 'H2', 'mH2', 'lightblue'),
        ('CO$_2$', 'CO2', 'mCO2', 'brown'),
        ('CO', 'CO', 'mCO', 'green'),
        ('SO$_2$', 'SO2', 'mSO2', 'red'),
        ('H$_2$S', 'H2S', 'mH2S', 'pink'),
        ('S$_2$', 'S2', 'mS2', 'orange'),
    ]

    def build_input_text(case):
        temperature = case.get('T', np.nan)
        delta_fmq = case.get('FMQ_sat', np.nan)
        h2o_ppm = case.get('mH2O_sat', np.nan)
        co2_ppm = case.get('mCO2_sat', np.nan)
        s_ppm = case.get('mS_sat', np.nan)
        feo_star_wt = case.get('FeO_star_wt', np.nan)

        temperature_c = temperature - 273.15 if np.isfinite(temperature) else np.nan
        h2o_ppm = h2o_ppm * 1e6 if np.isfinite(h2o_ppm) else np.nan
        co2_ppm = co2_ppm * 1e6 if np.isfinite(co2_ppm) else np.nan
        s_ppm = s_ppm * 1e6 if np.isfinite(s_ppm) else np.nan
        feo_star_pct = feo_star_wt * 100 if np.isfinite(feo_star_wt) else np.nan

        return '\n'.join([
            f'T = {temperature_c:.0f} C' if np.isfinite(temperature_c) else 'T = ?',
            rf'$f_{{\mathrm{{O_2}}}}^{{\mathrm{{sat}}}}$ = ' + f'FMQ{delta_fmq:+.2f}' if np.isfinite(delta_fmq) else r'$f_{\mathrm{O_2}}^{\mathrm{sat}}$ = ?',
            rf'$x_{{\mathrm{{H_2O}}}}^{{\mathrm{{sat}}}}$ = ' + f'{h2o_ppm:.0f} ppm wt' if np.isfinite(h2o_ppm) else r'$x_{\mathrm{H_2O}}^{\mathrm{sat}}$ = ?',
            rf'$x_{{\mathrm{{CO_2}}}}^{{\mathrm{{sat}}}}$ = ' + f'{co2_ppm:.0f} ppm wt' if np.isfinite(co2_ppm) else r'$x_{\mathrm{CO_2}}^{\mathrm{sat}}$ = ?',
            rf'$x_{{\mathrm{{S}}}}^{{\mathrm{{sat}}}}$ = ' + f'{s_ppm:.0f} ppm wt' if np.isfinite(s_ppm) else r'$x_{\mathrm{S}}^{\mathrm{sat}}$ = ?',
            rf'$x_{{\mathrm{{FeO*}}}}$ = ' + f'{feo_star_pct:.2f} wt%' if np.isfinite(feo_star_pct) else r'$x_{\mathrm{FeO*}}$ = ?',
        ])

    fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True, sharey=True)
    axes = axes.ravel()

    for i, (ax, case) in enumerate(zip(axes, cases), start=1):
        results = case['results']
        evo_df = case.get('evo_df')
        P_grid = results['P']

        for _, result_key, evo_key, color in species_styles:
            ax.loglog(P_grid, np.divide(results[result_key], P_grid), color=color)
            if evo_df is not None:
                ax.loglog(evo_df['P'], evo_df[evo_key], color=color, ls='--')

        ax.text(
            0.02,
            0.03,
            build_input_text(case),
            transform=ax.transAxes,
            ha='left',
            va='bottom',
            fontsize=inset_fontsize,
            bbox={'facecolor': 'white', 'edgecolor': '0.7', 'alpha': 0.9, 'boxstyle': 'round,pad=0.3'},
        )
        ax.set_xlim(1e-6, 1e2)
        ax.set_ylim(1e-3, 1)
        ax.yaxis.set_major_locator(mticker.LogLocator(base=10.0))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda value, _: f'{100 * value:g}%'))
        ax.yaxis.set_minor_formatter(mticker.NullFormatter())
        ax.tick_params(axis='both', which='major', labelsize=tick_fontsize)
        ax.tick_params(axis='both', which='minor', labelsize=tick_fontsize)
        ax.grid(True)

    subplot_left = 0.08
    subplot_right = 0.98
    subplot_center_x = 0.5 * (subplot_left + subplot_right)

    fig.text(subplot_center_x, 0.035, 'Total Pressure (bar)', ha='center', va='center', fontsize=axis_label_fontsize)
    fig.text(0.03, 0.5, 'Speciation in COHS gas (by moles)', ha='center', va='center', rotation='vertical', fontsize=axis_label_fontsize)

    style_handles = [
        Line2D([0], [0], color='black', lw=2, label='This work'),
        Line2D([0], [0], color='black', lw=2, ls='--', label='Liggins et al. 2020'),
    ]
    species_handles = [Line2D([0], [0], color=color, lw=2, label=label) for label, _, _, color in species_styles]
    fig.legend(handles=style_handles + species_handles, loc='upper center', bbox_to_anchor=(subplot_center_x, 0.975), borderaxespad=0, ncol=5, fontsize=legend_fontsize)
    fig.subplots_adjust(left=subplot_left, right=subplot_right, bottom=0.10, top=0.89, wspace=0.12, hspace=0.12)
    return fig, axes


In [ ]:
grid_case_specs = [
    {'mCO2_sat': 600e-6, 'mH2O_sat': 1000e-6, 'mS_sat': 1000e-6, 'FeO_star_wt': 10e-2, 'FMQ_sat': -1.5},
    {'mCO2_sat': 600e-6, 'mH2O_sat': 1000e-6, 'mS_sat': 1000e-6, 'FeO_star_wt': 10e-2, 'FMQ_sat': -0.01},
    {'mCO2_sat': 600e-6, 'mH2O_sat': 1000e-6, 'mS_sat': 1000e-6, 'FeO_star_wt': 10e-2, 'FMQ_sat': +1.5},
    {'mCO2_sat': 200e-6, 'mH2O_sat': 500e-6, 'mS_sat': 2000e-6, 'FeO_star_wt': 14.99e-2, 'FMQ_sat': -1.5},
    {'mCO2_sat': 200e-6, 'mH2O_sat': 500e-6, 'mS_sat': 2000e-6, 'FeO_star_wt': 14.99e-2, 'FMQ_sat': -0.01},
    {'mCO2_sat': 200e-6, 'mH2O_sat': 500e-6, 'mS_sat': 2000e-6, 'FeO_star_wt': 14.99e-2, 'FMQ_sat': +1.5},
]

grid_cases = []
for spec in grid_case_specs:
    model_results = calc_outgassing(
        T,
        spec['mH2O_sat'],
        spec['mCO2_sat'],
        spec['mS_sat'],
        spec['FMQ_sat'],
        alphaG_sat,
        spec['FeO_star_wt'],
        verbose=False,
    )
    try:
        evo_results = run_evo_simple(
            T,
            spec['mH2O_sat'],
            spec['mCO2_sat'],
            spec['mS_sat'],
            spec['FMQ_sat'],
            spec['FeO_star_wt'],
        )
    except Exception as exc:
        print(
            'EVo failed for '
            f"FMQ={spec['FMQ_sat']:+.2f}, H2O={spec['mH2O_sat'] * 1e6:.0f} ppm, "
            f"CO2={spec['mCO2_sat'] * 1e6:.0f} ppm, S={spec['mS_sat'] * 1e6:.0f} ppm, "
            f"FeO*={spec['FeO_star_wt'] * 100:.1f} wt%: {exc}"
        )
        evo_results = None
    grid_cases.append({
        'results': model_results,
        'evo_df': evo_results,
        'T': T,
        'FMQ_sat': spec['FMQ_sat'],
        'mH2O_sat': spec['mH2O_sat'],
        'mCO2_sat': spec['mCO2_sat'],
        'mS_sat': spec['mS_sat'],
        'FeO_star_wt': spec['FeO_star_wt'],
    })

plot_speciation_evo_grid(grid_cases)


In [ ]:
plot_speciation_evo_grid(grid_cases)